# Chạy toàn pipeline SDC theo thứ tự

Mỗi bước là một notebook, chạy từ trên xuống như bấm **Run All**, rồi lưu lại output vào chính file đó. Gặp lỗi thì **dừng ngay** ở bước lỗi — bước sau luôn đọc artifact của bước trước, chạy tiếp chỉ sinh ra kết quả sai lặng lẽ.

| # | Notebook | Đầu vào | Đầu ra |
|---|---|---|---|
| 00 | `00_common` | — | định nghĩa dùng chung, mọi bước tự nạp (không chạy riêng) |
| 01 | `predata/01_extract_cic` | pcap CIC-2022 `Data/1-Power`, `Data/2-Idle` | `Data/features/*_all.csv` |
| 02 | `predata/02_extract_capture` | pcap tự thu trong `CAPTURE_DIR` | `Data/features/*_capture.csv` |
| 03 | `predata/03_build_dataset` | `*_all.csv`, bảng nhãn | `Data/sessions_verified.parquet`, manifest |
| 04 | `predata/04_add_capture_devices` | `Data/sessions.parquet` + pcap tự thu | `sessions.parquet`, `sessions_verified.parquet` (có FIELD) |
| 05 | `train_model/05_train_closedset` | `sessions_verified.parquet` | `Models/<ts>_field_closedset/` |
| 06 | `train_model/06_train_openset` | `sessions_verified.parquet` | `Models/<run_id>/` (tiered) |
| 07 | `train_model/07_evaluate_openset` | run open-set | báo cáo time split / thiết bị lạ |
| 08 | `train_model/08_test_openset` | run open-set + IoT Sentinel | báo cáo Sentinel / CIC / FIELD |
| 09 | `out_model/09_build_sdc_iden` | closed-set mới nhất (05) | `Models/<ts>_iden/sdc_iden.onnx` — **bản deploy** |
| 10 | `out_model/10_activate_openset` | run open-set + báo cáo 07, 08 | `Models/current_model.json` (cờ tắt mặc định) |
| 11 | `out_model/11_export_onnx_openset` | run đang ghim | `sdc_multihead.onnx` |

Ba thư mục theo giai đoạn: `predata/` (01–04) dựng dữ liệu, `train_model/` (05–08) train và đo, `out_model/` (09–11) đóng gói model. Bản deploy chỉ cần **01–05 và 09**; 06–08, 10–11 là nhánh open-set để đánh giá.

## Chọn bước

`True` = chạy. Mặc định tắt `01_extract_cic` vì pcap thô CIC-2022 chỉ có trên server; bật lại khi chạy ở đó. Các cờ riêng của từng bước (`RUN_ID`, `ACTIVATE`...) vẫn nằm trong notebook của bước đó.

In [ ]:
STEPS = {
    "predata/01_extract_cic":           False,   # cần pcap thô CIC-2022 — chỉ có trên server
    "predata/02_extract_capture":       True,
    "predata/03_build_dataset":         True,
    "predata/04_add_capture_devices":   True,
    "train_model/05_train_closedset":   True,
    "train_model/06_train_openset":     False,   # nhánh open-set: bật khi cần đánh giá
    "train_model/07_evaluate_openset":  False,
    "train_model/08_test_openset":      False,
    "out_model/09_build_sdc_iden":      True,    # bản deploy
    "out_model/10_activate_openset":    False,
    "out_model/11_export_onnx_openset": False,
}
SAVE_OUTPUTS = True        # ghi output vào lại notebook, như Run All rồi Save
TIMEOUT_PER_CELL = None    # giây; None = không giới hạn (train có thể mất hàng giờ)

In [ ]:
import time
from pathlib import Path

import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

_cwd = Path.cwd().resolve()
PIPELINE = next((d for p in (_cwd, *_cwd.parents)
                 for d in (p, p / "pipeline", p / "Code" / "pipeline")
                 if (d / "00_common.ipynb").is_file()), None)
assert PIPELINE is not None, f"Không thấy pipeline/00_common.ipynb quanh {_cwd}"

missing = [s for s in STEPS if not (PIPELINE / f"{s}.ipynb").is_file()]
assert not missing, f"Thiếu notebook: {missing}"
plan = [s for s, on in STEPS.items() if on]
print("Pipeline:", PIPELINE)
print("Sẽ chạy :", " → ".join(plan) or "(không bước nào)")

In [ ]:
log = []
for step in plan:
    path = PIPELINE / f"{step}.ipynb"
    nb = nbformat.read(path, as_version=4)
    client = NotebookClient(nb, timeout=TIMEOUT_PER_CELL, kernel_name="python3",
                            resources={"metadata": {"path": str(path.parent)}})
    t0 = time.time()
    print(f"▶ {step} ...", flush=True)
    try:
        client.execute()
    except CellExecutionError as exc:
        if SAVE_OUTPUTS:   # giữ output tới cell lỗi để mở ra xem
            nbformat.write(nb, path)
        log.append((step, "LỖI", time.time() - t0))
        raise RuntimeError(f"Dừng ở {step}: mở {path.name} để xem cell lỗi") from exc
    if SAVE_OUTPUTS:
        nbformat.write(nb, path)
    log.append((step, "ok", time.time() - t0))
    print(f"  ok sau {time.time() - t0:.0f}s", flush=True)

for step, status, sec in log:
    print(f"{step:36s} {status:5s} {sec:8.0f}s")